1. Carga de librerias

In [71]:
import torch
import os
import pandas as pd
import random
import numpy as np
from google.colab import drive
from transformers import GPT2Tokenizer
from transformers import GPT2LMHeadModel
from torch.optim import AdamW
from torch.utils.data import DataLoader
from datetime import datetime

Configuración general del modelo

In [123]:
# CONFIGURACION GENERAL DEL PROYECTO

# Reproducibilidad
seed = 42

# Dataset
train_split = 0.90

# Tokenizacion
expected_vocab_size = 50257

# Secuencias
context_size = 256
batch_size = 16

# Modelo GPT from scratch
embed_dim = 128
num_heads = 4
num_layers = 4
dropout = 0.1

# Entrenamiento
learning_rate = 1e-4
max_epochs = 4
eval_interval = 200

# Dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"

print("CONFIGURACION CARGADA")
print(f"Device: {device}")
print(f"Context size: {context_size}")
print(f"Batch size: {batch_size}")
print(f"Learning rate: {learning_rate}")

CONFIGURACION CARGADA
Device: cuda
Context size: 256
Batch size: 16
Learning rate: 0.0001


2. Carga de datos

In [ ]:
drive.mount('/content/drive')
base = '/content/drive/MyDrive/Colab Notebooks/MAIA'
path = f'{base}/data/raw/nlp_llm_mp2/input.txt'

with open(path, 'r') as f:
    corpus = f.read()

print(corpus[:500])

Mounted at /content/drive
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


Validación inicial del corpus

In [ ]:
print("VALIDACION INICIAL DEL CORPUS")

print(f"Total de caracteres: {len(corpus):,}")

words = corpus.split()
print(f"Total aproximado de palabras: {len(words):,}")

lines = corpus.splitlines()
print(f"Total de lineas: {len(lines):,}")

print("\nPrimeras 5 lineas:")
for i in range(5):
    print(f"{i+1}: {lines[i]}")

print("\nUltimas 5 lineas:")
for i in range(5):
    print(f"{len(lines)-4+i}: {lines[-5+i]}")

VALIDACION INICIAL DEL CORPUS
Total de caracteres: 1,115,394
Total aproximado de palabras: 202,651
Total de lineas: 40,000

Primeras 5 lineas:
1: First Citizen:
2: Before we proceed any further, hear me speak.
3: 
4: All:
5: Speak, speak.

Ultimas 5 lineas:
39996: 
39997: ANTONIO:
39998: Noble Sebastian,
39999: Thou let'st thy fortune sleep--die, rather; wink'st
40000: Whiles thou art waking.


Tokenización del corpus

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

print("VALIDACION DEL TOKENIZER GPT2")

vocab_size = tokenizer.vocab_size
print(f"Tamaño del vocabulario: {vocab_size}")

tokens = tokenizer.encode(
    corpus,
    add_special_tokens=False
)

print(f"Total de tokens del corpus: {len(tokens):,}")

print("\nPrimeros 20 token IDs:")
print(tokens[:20])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

VALIDACION DEL TOKENIZER GPT2
Tamaño del vocabulario: 50257


Token indices sequence length is longer than the specified maximum sequence length for this model (338025 > 1024). Running this sequence through the model will result in indexing errors


Total de tokens del corpus: 338,025

Primeros 20 token IDs:
[5962, 22307, 25, 198, 8421, 356, 5120, 597, 2252, 11, 3285, 502, 2740, 13, 198, 198, 3237, 25, 198, 5248]


Validación del tamaño del corpus

In [ ]:
actual_vocab_size = tokenizer.vocab_size

assert actual_vocab_size == expected_vocab_size

División del conjunto de datos: Train / Validation

In [ ]:
split_idx = int(train_split * len(tokens))

train_tokens = tokens[:split_idx]
val_tokens = tokens[split_idx:]

print("DIVISION TRAIN VALIDATION")

print(f"Total de tokens: {len(tokens):,}")
print(f"Tokens de entrenamiento: {len(train_tokens):,}")
print(f"Tokens de validacion: {len(val_tokens):,}")

print(f"\nPorcentaje train: {len(train_tokens)/len(tokens):.2%}")
print(f"Porcentaje validation: {len(val_tokens)/len(tokens):.2%}")

DIVISION TRAIN VALIDATION
Total de tokens: 338,025
Tokens de entrenamiento: 304,222
Tokens de validacion: 33,803

Porcentaje train: 90.00%
Porcentaje validation: 10.00%


Convertimos a tensor

In [ ]:
train_data = torch.tensor(train_tokens, dtype=torch.long)
val_data = torch.tensor(val_tokens, dtype=torch.long)

print("DATOS CONVERTIDOS A TENSOR")

print(f"Train shape: {train_data.shape}")
print(f"Validation shape: {val_data.shape}")
print(f"Tipo de dato: {train_data.dtype}")

DATOS CONVERTIDOS A TENSOR
Train shape: torch.Size([304222])
Validation shape: torch.Size([33803])
Tipo de dato: torch.int64


Funcion que realiza el desplazamiento de tokens para el entrenamiento (batches autoregresivos)

In [ ]:
"""Función que realiza el desplazamiento de tokens para el entrenamiento
utilizada para la construcción del modelo desde cero"""

def get_batch(split):
    data = train_data if split == "train" else val_data

    ix = torch.randint(
        len(data) - context_size,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i + context_size]
        for i in ix
    ])

    y = torch.stack([
        data[i + 1:i + context_size + 1]
        for i in ix
    ])

    x = x.to(device)
    y = y.to(device)

    return x, y

In [ ]:
x_batch, y_batch = get_batch("train")

print("VALIDACION DE BATCHES")

print(f"x shape: {x_batch.shape}")
print(f"y shape: {y_batch.shape}")

print("\nPrimer ejemplo de entrada (x):")
print(x_batch[0])

print("\nPrimer ejemplo target (y):")
print(y_batch[0])

VALIDACION DE BATCHES
x shape: torch.Size([16, 128])
y shape: torch.Size([16, 128])

Primer ejemplo de entrada (x):
tensor([ 3656,   329, 10443,    25,   611,   428,  1705,   307,  2081,    11,
          198, 43920, 16599,   290,  3367,    11,   534, 10515,   318,   475,
         2626,    26,   198,  1890, 49398,   318,   257, 11800,   393,  1352,
           11,   198,  1870, 10174,   257, 19716,  2582,  1839,   351,  3867,
         2456,    13,   198,  3886,   428,  1848,   788, 19579,   743,  1592,
          683,    26,   198,  1890,   673,   338,   257,  2415,   284,   307,
         6028,   798,   881,    25,   198,  9360, 19680,    82,   481,   787,
          257,  6555,   287,   465,  9296,    26,   198,  9360, 10953,   481,
          279,  9798,   656,   257, 30623,  2612,    26,   198,   464, 26241,
          481,   307, 11607,   348,  2915,   673,   288,   849, 25722,    26,
          198,  1870, 46810,   481,   307, 36912,   351, 34081,    11,   198,
         2514,  3285,   29

3. Modelo 1: Fine-tuning GPT2

: Carga de modelo pre-entrenado

In [124]:
print("CARGA DEL MODELO GPT2")

model_gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
model_gpt2 = model_gpt2.to(device)

print("Modelo cargado correctamente")
print(f"Device: {device}")

CARGA DEL MODELO GPT2


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo cargado correctamente
Device: cuda


Creación de la clase para construir los dataset para el modelo GPT2 pre-entrenado

In [125]:
from torch.utils.data import Dataset

class GPT2Dataset(Dataset):
    def __init__(self, tokens, context_size):
        self.input_ids = []

        for i in range(0, len(tokens) - context_size, context_size):
            block = tokens[i:i + context_size]
            self.input_ids.append(torch.tensor(block, dtype=torch.long))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        input_ids = self.input_ids[idx]

        return {
            "input_ids": input_ids,
            "labels": input_ids.clone()
        }

Creación de los dataset

In [126]:
train_dataset = GPT2Dataset(train_tokens, context_size)
val_dataset = GPT2Dataset(val_tokens, context_size)

print("DATASETS GPT2 CREADOS")

print(f"Bloques de entrenamiento: {len(train_dataset):,}")
print(f"Bloques de validacion: {len(val_dataset):,}")

DATASETS GPT2 CREADOS
Bloques de entrenamiento: 1,188
Bloques de validacion: 132


In [127]:
sample = train_dataset[0]

print("VALIDACION DEL DATASET GPT2")

print(f"Input shape: {sample['input_ids'].shape}")
print(f"Labels shape: {sample['labels'].shape}")

print("\nPrimer bloque input_ids:")
print(sample["input_ids"])

print("\nPrimer bloque labels:")
print(sample["labels"])

VALIDACION DEL DATASET GPT2
Input shape: torch.Size([256])
Labels shape: torch.Size([256])

Primer bloque input_ids:
tensor([ 5962, 22307,    25,   198,  8421,   356,  5120,   597,  2252,    11,
         3285,   502,  2740,    13,   198,   198,  3237,    25,   198,  5248,
          461,    11,  2740,    13,   198,   198,  5962, 22307,    25,   198,
         1639,   389,   477, 12939,  2138,   284,  4656,   621,   284,  1145,
          680,    30,   198,   198,  3237,    25,   198,  4965,  5634,    13,
        12939,    13,   198,   198,  5962, 22307,    25,   198,  5962,    11,
          345,   760,   327,  1872,   385,  1526, 28599,   318,  4039,  4472,
          284,   262,   661,    13,   198,   198,  3237,    25,   198,  1135,
          760,   470,    11,   356,   760,   470,    13,   198,   198,  5962,
        22307,    25,   198,  5756,   514,  1494,   683,    11,   290,   356,
         1183,   423, 11676,   379,   674,   898,  2756,    13,   198,  3792,
          470,   257, 155

Construcción del dataloader

In [128]:
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("DATALOADERS CREADOS")

print(f"Seed utilizada: {seed}")
print(f"Batch size: {batch_size}")
print(f"Train batches: {len(train_loader):,}")
print(f"Validation batches: {len(val_loader):,}")

DATALOADERS CREADOS
Seed utilizada: 42
Batch size: 16
Train batches: 75
Validation batches: 9


In [129]:
batch = next(iter(train_loader))

print("VALIDACION DEL DATALOADER")

print(f"input_ids shape: {batch['input_ids'].shape}")
print(f"labels shape: {batch['labels'].shape}")

VALIDACION DEL DATALOADER
input_ids shape: torch.Size([16, 256])
labels shape: torch.Size([16, 256])


Guardado de modelo (primera vez - creacion del archivo)

In [57]:
log_path = "experiments_log.csv"

if not os.path.exists(log_path):
    df_log = pd.DataFrame(columns=[
        "version",
        "date",
        "model",
        "context_size",
        "batch_size",
        "learning_rate",
        "max_epochs",
        "train_split",
        "validation_split",
        "device",
        "final_train_loss",
        "best_validation_loss",
        "best_epoch",
        "overfitting_detected",
        "checkpoint_path",
        "notes"
    ])

    df_log.to_csv(log_path, index=False)

print("LOG DE EXPERIMENTOS INICIALIZADO")
print(f"Archivo: {log_path}")

LOG DE EXPERIMENTOS INICIALIZADO
Archivo: experiments_log.csv


👉 Definición del optimizer y entrenamiento del modelo

In [130]:

optimizer = AdamW(
    model_gpt2.parameters(),
    lr=learning_rate,
    weight_decay=0.001
)

print("OPTIMIZER CONFIGURADO")
print(f"Learning rate: {learning_rate}")

best_val_loss = float("inf")
best_epoch = 0
final_train_loss = 0
train_loss_at_best_epoch = 0

df_log = pd.read_csv(log_path)
next_version = f"v{len(df_log) + 1}"
checkpoint_path = f"gpt2_finetuned_{next_version}.pt"

for epoch in range(max_epochs):
    model_gpt2.train()
    total_train_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_gpt2(
            input_ids=input_ids,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    final_train_loss = avg_train_loss

    model_gpt2.eval()
    total_val_loss = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            outputs = model_gpt2(
                input_ids=input_ids,
                labels=labels
            )

            loss = outputs.loss
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch = epoch + 1
        train_loss_at_best_epoch = avg_train_loss

        torch.save(
            model_gpt2.state_dict(),
            checkpoint_path
        )

        print(f"Nuevo mejor modelo guardado en epoch {best_epoch}")

    print(
        f"Epoch {epoch + 1}/{max_epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Validation Loss: {avg_val_loss:.4f}"
    )

print("\nMEJOR RESULTADO")
print(f"Best Validation Loss: {best_val_loss:.4f}")
print(f"Best Epoch: {best_epoch}")
print(f"Train Loss at Best Epoch: {train_loss_at_best_epoch:.4f}")
print(f"Best Checkpoint: {checkpoint_path}")

OPTIMIZER CONFIGURADO
Learning rate: 0.0001
Nuevo mejor modelo guardado en epoch 1
Epoch 1/4 | Train Loss: 3.7379 | Validation Loss: 3.3687
Nuevo mejor modelo guardado en epoch 2
Epoch 2/4 | Train Loss: 3.4087 | Validation Loss: 3.3304
Nuevo mejor modelo guardado en epoch 3
Epoch 3/4 | Train Loss: 3.2663 | Validation Loss: 3.3198
Epoch 4/4 | Train Loss: 3.1423 | Validation Loss: 3.3209

MEJOR RESULTADO
Best Validation Loss: 3.3198
Best Epoch: 3
Train Loss at Best Epoch: 3.2663
Best Checkpoint: gpt2_finetuned_v10.pt


Guardado de experimento

In [131]:
df_log = pd.read_csv(log_path)

next_version = f"v{len(df_log) + 1}"
checkpoint_path = f"gpt2_finetuned_{next_version}.pt"

torch.save(
    model_gpt2.state_dict(),
    checkpoint_path
)

new_experiment = {
    "version": next_version,
    "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model": "GPT2 Fine-Tuning",
    "context_size": context_size,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "max_epochs": max_epochs,
    "train_split": train_split,
    "validation_split": 1 - train_split,
    "device": device,
    "final_train_loss": final_train_loss,
    "best_validation_loss": best_val_loss,
    "best_epoch": best_epoch,
    "overfitting_detected": "Yes" if best_val_loss > train_loss_at_best_epoch else "No",
    "checkpoint_path": checkpoint_path,
    "notes": "Primer experimento estructural sobre longitud de contexto manteniendo baseline optimizado"
}

df_log.loc[len(df_log)] = new_experiment
df_log.to_csv(log_path, index=False)

print("EXPERIMENTO REGISTRADO")
print(df_log.tail(1))

EXPERIMENTO REGISTRADO
  version                 date             model  context_size  batch_size  \
9     v10  2026-04-21 20:02:22  GPT2 Fine-Tuning           256          16   

   learning_rate  max_epochs  train_split  validation_split device  \
9         0.0001           4          0.9               0.1   cuda   

   final_train_loss  best_validation_loss  best_epoch overfitting_detected  \
9          3.142304              3.319813           3                  Yes   

         checkpoint_path                                              notes  
9  gpt2_finetuned_v10.pt  Primer experimento estructural sobre longitud ...  


Generación de Texto (Inferencia)

In [132]:
def generate_text(
    model,
    tokenizer,
    prompt,
    max_new_tokens=100,
    temperature=0.8,
    top_k=50,
    device=device
):
    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return generated_text

print("FUNCION DE GENERACION CARGADA")

FUNCION DE GENERACION CARGADA


In [133]:
tokenizer.pad_token = tokenizer.eos_token

print("PAD TOKEN CONFIGURADO")
print(f"Pad token: {tokenizer.pad_token}")
print(f"Pad token id: {tokenizer.pad_token_id}")

PAD TOKEN CONFIGURADO
Pad token: <|endoftext|>
Pad token id: 50256


In [134]:
prompts = [
    "ROMEO:",
    "KING:",
    "To be, or not to be"
]

print("GENERACION DE TEXTO\n")

for i, prompt in enumerate(prompts, 1):
    generated = generate_text(
        model=model_gpt2,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=120,
        temperature=0.8,
        top_k=50
    )

    print(f"Ejemplo {i}")
    print(f"Prompt: {prompt}")
    print(generated)
    print("\n" + "-" * 80 + "\n")

GENERACION DE TEXTO

Ejemplo 1
Prompt: ROMEO:
ROMEO:
I'll give you my reasons:
I am not to be accused of false,
But the matter shall be mine.

DUKE VINCENTIO:
But what is your reason?

DUKE VINCENTIO:
My reason is to be so brief.

DUKE VINCENTIO:
Because I know well what you mean.

DUKE VINCENTIO:
This is a reason.

DUKE VINCENTIO:
And the reason is,
To get you mad with your

--------------------------------------------------------------------------------

Ejemplo 2
Prompt: KING:
KING:
Be it as it may.

ROMEO:
And as thou hast done, O,
I humbly thank thy grace.

First Murderer:
I thank thee for imprison'd you, for you
were not in prison.

Second Murderer:
And as thou hast done,
My death hath been my doom.

ROMEO:
O, sir, I will die, and then thou
shall forget thy sins.

First Murderer:
Go and find a way to forget them,
And that thou mayst not do

--------------------------------------------------------------------------------

Ejemplo 3
Prompt: To be, or not to be
To be, or not to be,


Reinicio de log (reseteo del log de experimentos)

In [49]:
log_path = "experiments_log.csv"

if os.path.exists(log_path):
    os.remove(log_path)
    print("Archivo de log anterior eliminado")

df_log = pd.DataFrame(columns=[
    "version",
    "date",
    "model",
    "context_size",
    "batch_size",
    "learning_rate",
    "max_epochs",
    "train_split",
    "validation_split",
    "device",
    "final_train_loss",
    "best_validation_loss",
    "best_epoch",
    "overfitting_detected",
    "checkpoint_path",
    "notes"
])

df_log.to_csv(log_path, index=False)

print("LOG REINICIADO CORRECTAMENTE")
print(f"Nuevo archivo creado: {log_path}")

Archivo de log anterior eliminado
LOG REINICIADO CORRECTAMENTE
Nuevo archivo creado: experiments_log.csv
